In [1]:
import os

for var in [
    "MASTER",
    "SPARK_MASTER",
    "PYSPARK_SUBMIT_ARGS",
    "SPARK_REMOTE",
    "SPARK_CONNECT_MODE_ENABLED"
]:
    print(var, "ANTES =", os.environ.get(var))
    os.environ.pop(var, None)
    print(var, "DESPUÉS =", os.environ.get(var))

MASTER ANTES = None
MASTER DESPUÉS = None
SPARK_MASTER ANTES = None
SPARK_MASTER DESPUÉS = None
PYSPARK_SUBMIT_ARGS ANTES = None
PYSPARK_SUBMIT_ARGS DESPUÉS = None
SPARK_REMOTE ANTES = None
SPARK_REMOTE DESPUÉS = None
SPARK_CONNECT_MODE_ENABLED ANTES = None
SPARK_CONNECT_MODE_ENABLED DESPUÉS = None


In [1]:
import os
import sys

# Limpiar variables problemáticas
for var in [
    "MASTER",
    "SPARK_MASTER",
    "PYSPARK_SUBMIT_ARGS",
    "SPARK_REMOTE",
    "SPARK_CONNECT_MODE_ENABLED"
]:
    os.environ.pop(var, None)

# Forzar el Python correcto
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("test")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .config("spark.executorEnv.PYSPARK_PYTHON", r"C:\Users\Theas\AppData\Local\Programs\Python\Python311\python.exe")
    .getOrCreate()
)

sc = spark.sparkContext
print("master:", sc.master)
print("python:", sys.executable)

master: local[1]
python: C:\Users\Theas\AppData\Local\Programs\Python\Python311\python.exe


In [3]:
print(sc.parallelize([1, 2, 3, 4]).map(lambda x: x * 2).collect())

[2, 4, 6, 8]


In [5]:
# Datos de entrada: (fila, columna, valor)
A_data = [
    (0, 0, 1),
    (0, 1, 2),
    (1, 0, 3),
    (1, 1, 4)
]

B_data = [
    (0, 0, 5),
    (0, 1, 6),
    (1, 0, 7),
    (1, 1, 8)
]

# Crear RDDs
A = sc.parallelize(A_data)
B = sc.parallelize(B_data)

# MAP:
# A(i,j,aij) -> (j, ("A", i, aij))
# B(j,k,bjk) -> (j, ("B", k, bjk))
A_by_j = A.map(lambda x: (x[1], ("A", x[0], x[2])))
B_by_j = B.map(lambda x: (x[0], ("B", x[1], x[2])))

# GROUP BY KEY:
# Agrupar por la dimensión compartida j
grouped = A_by_j.union(B_by_j).groupByKey()

# REDUCE lógico:
# Generar productos parciales
def generate_partial_products(record):
    j, values = record
    values = list(values)

    a_values = []
    b_values = []

    for matrix_name, index, value in values:
        if matrix_name == "A":
            a_values.append((index, value))   # (i, aij)
        else:
            b_values.append((index, value))   # (k, bjk)

    partials = []
    for i, aij in a_values:
        for k, bjk in b_values:
            partials.append(((i, k), aij * bjk))

    return partials

partial_products = grouped.flatMap(generate_partial_products)

# Sumar productos parciales por cada posición (i,k)
C = partial_products.reduceByKey(lambda x, y: x + y)

# Mostrar resultado ordenado
resultado = C.sortByKey().collect()
print("Resultado ordenado:", resultado)

# Convertir a matriz 2x2
resultado_dict = dict(resultado)
matriz_resultado = [
    [resultado_dict[(0, 0)], resultado_dict[(0, 1)]],
    [resultado_dict[(1, 0)], resultado_dict[(1, 1)]]
]

print("Matriz resultante:")
for fila in matriz_resultado:
    print(fila)

Resultado ordenado: [((0, 0), 19), ((0, 1), 22), ((1, 0), 43), ((1, 1), 50)]
Matriz resultante:
[19, 22]
[43, 50]
